In [180]:
import pandas as pd
import numpy as np
import math
from tqdm import tqdm
# add path so we can import my package
import sys
sys.path.append("/Users/willneuner/Desktop/FINTECH545") 

In [2]:
from scipy.stats import norm, t

In [3]:
# DISTRIBUTIONS (NORMAL AND T)
from risk_management import distributions

# mu_vector, covariance_matrix = fit_multivariate_normal_dist(x: pd.DataFrame)

# mu, sigma, nu = fit_univariate_t_dist(x)

# alpha, betas, mu, sigma, nu = t_regression(X: pd.DataFrame, y:pd.Series, add_constant:bool = True, print_summary = False)

In [4]:
# MEASUREMENTS (CORRELATION, COVARIANCE, FACTORIZATIONS, RETURNS)
from risk_management import measurements

# corr = compute_correlation(x:pd.DataFrame, method="pearson", drop_missing = False, exponentially_weighted = False, lambda_ = 0.97, ddof: int =1)

# cov = compute_covariance(x:pd.DataFrame, drop_missing = False, exponentially_weighted = False, lambda_ = 0.97, ddof:int=1)

# cov = compute_covariance_with_ew_corr(x: pd.DataFrame, corr_lambda, var_lambda)

# psd = near_psd(A: pd.DataFrame, epsilon = 0.0)
# psd = higham_psd(A: pd.DataFrame, tolerance = 1e-8, max_iterations= 100_000)

# A = cholesky_factor(x:pd.DataFrame)

# returns = compute_returns(x: pd.DataFrame, return_type = "arithmetic")


In [5]:
# RISK METRICS (VAR, ES)
from risk_management import risk_metrics

# NOTE -> all of these assume that X was a set of RETURNS, not prices

# abs_VaR, rel_VaR = univariate_normal_VaR(mean: float, std: float, alpha = 0.05)

# nu, mu, sigma = t.fit(x)
# abs_VaR, rel_VaR = univariate_t_VaR(mu: float, sigma: float, nu: float, alpha: float = 0.05)

# abs_ES, diff_ES = expected_shortfall_normal(mu:float, sigma: float, alpha = 0.05)

# mu, sigma, nu = fit_univariate_t_dist(x)
# abs_ES, diff_ES = expected_shortfall_t(mu: float, sigma: float, nu: float, alpha: float = 0.05)



In [6]:
## SIMULATIONS
from risk_management import simulations

# NOTE -> all of these assume that X was a set of RETURNS, not prices

# simulation_data = normal_monte_carlo_simulation(mean_vector, covariance_matrix, n_sims, fix_method, seed=1234) # len(cov), n_sims

# simulation_data = pca_monte_carlo_simulation(mean_vector, covariance_matrix, n_sims, explained_threshold = 0.99, seed=1234) # len(cov), n_sims

# current_prices = vector of start prices of assets, holdings = vector of how many of each asset we have
# abs_VaR, rel_VaR = monte_carlo_VaR_sim(mean_vector, covariance_matrix, current_prices, holdings, n_draws, return_type = "arithmetic", alpha = 0.05, seed = 1234) 

# sim_dataframe = VaR_ES_2_level_sim_from_copula(sample_data: pd.DataFrame, holdings: np.array, prices: np.array, fix_method, n_sims = 100_000, alpha=0.05, seed=1234)

In [7]:
from risk_management import goodness_of_fit

# r2, adjr2, Aic, Aicc, BIC

from risk_management import asset_pricing

# European, American binary, American Discontinuous div

from risk_management import portfolio_construction

# risk parity, weighted risk parity, max sharpe, efficient frontier, 

from risk_management import risk_attribution
# ex_post attribution, ex_post factor attribution


### Question 1

- Risk analysis -> we care more about distribution of outcomes and worst case scenario
- Forecasting -> we care more about expected value and mean future outcome

### Question 2

In [17]:
data_2 = pd.read_csv("problem2.csv")

# 2a
data_2.mean(), data_2.var(), data_2.skew(), data_2.kurt()

# 2b
# I would choose T because the excess kurtosis is greater than 0, so there are fatter tails than one would see from just a normal distribution

# 2c
mean, var = distributions.fit_multivariate_normal_dist(data_2)
mu, sigma, nu = distributions.fit_univariate_t_dist(data_2)

n = len(data_2)
params_norm = {"loc": mean, "scale": np.sqrt(var)}

LL_norm = goodness_of_fit.log_likelihood(norm.pdf, params_norm, data_2)
aicc_norm = goodness_of_fit.aicc(2, LL_norm, n)

params_t = {"loc": mean, "scale": sigma, "df":nu}
LL_t = goodness_of_fit.log_likelihood(t.pdf, params_t, data_2)
aicc_t = goodness_of_fit.aicc(3, LL_t, n)

bic_norm = goodness_of_fit.bic(2, LL_norm, n)
bic_t = goodness_of_fit.bic(3, LL_t, n)
# print(bic_norm, bic_t)
# lower criterion is better

# since the t distribution information criteria was lower, we think that is the better model

### Question 3

In [27]:
data_3 = pd.read_csv("problem3.csv")

cov = measurements.compute_covariance(data_3, drop_missing=False)

eigvals, _ = np.linalg.eigh(cov)
print(eigvals)
# eigenvalues are not all non-negative -> not positive semi definite

cov_fixed = measurements.higham_psd(cov, tolerance=1e-20)
eigvals, _ = np.linalg.eigh(cov_fixed)
eigvals

[-0.31024286 -0.13323183  0.02797828  0.83443367  6.78670573]


array([1.66573288e-16, 9.33125467e-16, 2.16398622e-15, 7.36663625e-01,
       6.46897936e+00])

### Question 4

In [33]:
data_4 = pd.read_csv("problem4.csv")
cov = measurements.compute_covariance(data_4, exponentially_weighted=True, lambda_=0.94)

# parity weights with std dev
weights = portfolio_construction.compute_risk_parity_weights(cov, risk_budgets=np.ones(cov.shape[0]), pos_weights=True)
print(weights)

# parity weights with ES
def min_sse_ces(w, cov, risk_budgets, alpha=0.05):
    z_alpha = norm.ppf(alpha)
    ev_given_z_lt_z_alpha = (-norm.pdf(z_alpha) / alpha)

    m = cov @ w
    vol = np.sqrt(w.T @ cov @ w)
    
    ES = -(vol * ev_given_z_lt_z_alpha)
    cES = w * m * ev_given_z_lt_z_alpha / vol
    cES_budgeted = cES / risk_budgets
    return np.sum((cES_budgeted - np.mean(cES_budgeted))**2)

risk_budgets = np.array([1,1,1,1,1])
ES_risk_parity_weights = portfolio_construction.compute_risk_parity_weights(cov, risk_budgets, custom_objective_func=lambda x, y, z: min_sse_ces(x, y, z, alpha=0.05))
print(ES_risk_parity_weights)

# the expected shortfall of the 5% quantile of the portfolio returns assuming multivariate normality
# is a linear function of standard deviation.  Therefore the risk parity weights are the same.

Optimization terminated successfully    (Exit mode 0)
            Current function value: 4.936181344411211e-18
            Iterations: 41
            Function evaluations: 250
            Gradient evaluations: 41
[0.08325191 0.0829683  0.21313211 0.42411307 0.19653461]
Optimization terminated successfully    (Exit mode 0)
            Current function value: 9.184666330318844e-18
            Iterations: 29
            Function evaluations: 187
            Gradient evaluations: 29
[0.08325193 0.08296831 0.21313213 0.42411305 0.19653458]


### Question 5

In [47]:
data_5 = pd.read_csv("problem5.csv")

weights = np.array([0.3, 0.2, 0.5])

(portfolio_returns, 
total_portfolio_return, 
asset_returns, k_t, 
weights_over_time, 
attributed_returns,  
risk_attributions, 
portfolio_vol) = risk_attribution.ex_post_attribution(data_5.to_numpy(), weights)

attributed_returns, risk_attributions

(array([-0.06551252, -0.00221981,  0.14883061]),
 array([-0.00062022,  0.0028271 ,  0.01257982]))

### Question 6

In [71]:
data_6 = pd.read_csv("problem6.csv")

# arithmetic returns
# 0 mean
rfr = 0.0475
days = 252
# iv is constant
# report var and $ at 5% value

returns = measurements.compute_returns(data_6, return_type="arithmetic").drop(columns="Date")

In [72]:
# fit each asset for normally distributed returns
means, cov = distributions.fit_multivariate_normal_dist(returns)
means = means.to_numpy()
cov = cov.to_numpy()
means, cov

(array([0.00038432, 0.00237336]),
 array([[0.0001815 , 0.00010329],
        [0.00010329, 0.00022892]]))

#### C: Delta Normal

In [95]:
from scipy.optimize import root_scalar
def implied_vol_gbsm(price, S, X, T, r, b, option_type="call"):
    option_type = str.lower(option_type)
    if option_type not in ["call", "put"]:
        raise ValueError("Option type must be call or put")
    
    def obj(sigma):
        value, *_ = asset_pricing.European_GBSM(S, X, T, sigma, r, b, option_type)
        return value - price
    
    # bracket between [-1e-20, 5] (500% vol upper bound)
    sol = root_scalar(obj, bracket=[-1e-20, 10], method='bisect')
    return sol.root

In [155]:
stock_prices = data_6.drop(columns="Date").iloc[-1].to_numpy()
# stock_prices

# Stock A American put price with X = 100, TTM = 1 year, vol = 0.2, dividend = 0.025 60 days and 220 days from now
X = 100
T = 1
vol = 0.2
div_amts = [0.025, 0.025]
div_times = [60/days, 220/days]
# lcm_multiple = math.lcm(60, 160) # I could do this instead, right?
# print(lcm_multiple)
# lcm_multiple = math.lcm(60/days, 220/days)
# N = lcm_multiple // 2
N = 252
put_0_price = asset_pricing.American_Binary_Tree_Discontinuous_Div(stock_prices[0], X, T, vol, rfr, div_amts=[0.025, 0.025], div_times=div_times, N=N, option_type="put")
print(put_0_price)
d_price = 0.001
put_0_price_ddown = asset_pricing.American_Binary_Tree_Discontinuous_Div(stock_prices[0]-d_price, X, T, vol, rfr, div_amts=[0.025, 0.025], div_times=div_times, N=N, option_type="put")
put_0_price_dup = asset_pricing.American_Binary_Tree_Discontinuous_Div(stock_prices[0]+d_price, X, T, vol, rfr, div_amts=[0.025, 0.025], div_times=div_times, N=N, option_type="put")
put_0_delta = (put_0_price_dup - put_0_price_ddown) / (2*d_price)

# Stock B European call
X = 100
call_1_price = 6.25
T = 100 / days
div = 0 # no dividends
b = rfr-div # no dividends
i_vol = implied_vol_gbsm(call_1_price, stock_prices[1], X, T, rfr, b, option_type="call")
_, call_1_delta, *_ = asset_pricing.European_GBSM(stock_prices[1], X, T, i_vol, rfr, b, option_type="call")

asset_prices = np.array([stock_prices[0], stock_prices[1], put_0_price, call_1_price])

6.177451402646503


In [128]:
underlying_prices = stock_prices
price_indices = np.array([0, 1, 0, 1]) # what each asset's underlying is
holdings = np.array([100, 50, 100, -50])
portfolio_value = holdings @ asset_prices
weights = holdings * asset_prices / portfolio_value

deltas = np.array([1, 1, put_0_delta, call_1_delta])

In [131]:
VaR = risk_metrics.delta_normal_var(asset_prices, weights, underlying_prices, price_indices, deltas, cov, alpha=0.05)
VaR

np.float64(1.021970445454031)

In [181]:
ES = risk_metrics.delta_normal_es(asset_prices, weights, underlying_prices, price_indices, deltas, cov, alpha=0.05)
ES

np.float64(1.2815921685621758)

#### D: Monte Carlo

In [209]:
sim_returns = simulations.normal_monte_carlo_simulation(np.array([0.0, 0.0]), cov, n_sims=100, fix_method=measurements.higham_psd).T
sim_portfolio_values = []
alpha = 0.05
# portfolio_value
i_vol = implied_vol_gbsm(call_1_price, stock_prices[1], X, T, rfr, b, option_type="call")

for returns in tqdm(sim_returns):
    new_underlying_prices = stock_prices * (1+returns)
    
    X = 100
    T = 251/days
    vol = 0.2
    div_amts = [0.025, 0.025]
    div_times = [59/days, 219/days]
    N = 251
    new_put_0_price = asset_pricing.American_Binary_Tree_Discontinuous_Div(new_underlying_prices[0], X, T, vol, rfr, div_amts=[0.025, 0.025], div_times=div_times, N=N, option_type="put")

    X = 100
    T = 99 / days
    div = 0 # no dividends
    b = rfr-div # no dividends
    
    new_call_1_price, *_ = asset_pricing.European_GBSM(new_underlying_prices[1], X, T, i_vol, rfr, b, option_type="call")
    new_prices = np.array([new_underlying_prices[0], new_underlying_prices[1], new_put_0_price, new_call_1_price])
    
    new_portfolio_value = new_prices @ holdings
    sim_portfolio_values.append(new_portfolio_value)

sim_portfolio_values = np.array(sim_portfolio_values)
sorted_values = np.sort(sim_portfolio_values)
percentile_portfolio = np.percentile(sorted_values, 100 * alpha)
abs_VaR = portfolio_value - percentile_portfolio

below_alpha_portfolios = sorted_values[sorted_values <= np.percentile(sorted_values, 100 * alpha)]
portfolios_es = portfolio_value - below_alpha_portfolios
abs_ES = np.mean(portfolios_es)











































































































































































































100%|██████████| 100/100 [30:24<00:00, 18.25s/it]


In [213]:
portfolio_value = new_prices @ holdings

In [219]:
portfolio_value - np.percentile((sorted_values), 5)

np.float64(135.18307013240155)

In [212]:
abs_VaR, abs_ES

(np.float64(142.75389011995685), np.float64(169.8239560799324))

In [221]:
below_alpha_portfolios - portfolio_value

array([-208.33595197, -166.89413425, -154.8451097 , -142.66494202,
       -138.52554252])